# Interactive Novel Video Generator Pipeline

This notebook allows you to control the entire video generation pipeline step-by-step using the unified CLI. 
Each step runs as a standalone command, updating you on the progress.

## 0. Setup and Configuration
Define your chapter file and output directories here. This configuration is used by all subsequent steps.

In [ ]:
import os
import json
from pathlib import Path

# --- FIX WORKING DIRECTORY & PATHS ---
# Ensure we are running from the project root, not the notebooks directory
current_dir = Path(os.getcwd())
if current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
    os.chdir(PROJECT_ROOT)
    print(f"Changed working directory to: {os.getcwd()}")
else:
    PROJECT_ROOT = current_dir

# Normalize for shell usage (forward slashes)
PROJECT_ROOT_STR = str(PROJECT_ROOT).replace("\\", "/")

# --- CONFIGURATION ---
# Path to the input chapter file
CHAPTER_FILE_PATH = f"{PROJECT_ROOT_STR}/data/ihacw/chapters/ihacw_ch0001.json"

# Base output directory
OUTPUT_BASE = f"{PROJECT_ROOT_STR}/outputs/interactive_run"
# ---------------------

# Helper to extract ID roughly or just use filename
chapter_name = Path(CHAPTER_FILE_PATH).stem # e.g. "ihacw_ch0001"
# We assume the script extracts ID from content, but we need it for paths below
# Let's try to peek at the file to get the real ID to be safe for our variables
try:
    with open(CHAPTER_FILE_PATH, 'r', encoding='utf-8') as f:
        data = json.load(f)
        CHAPTER_ID = data.get('chapter_number', data.get('id', 'unknown'))
except Exception as e:
    print(f"Warning: Could not read chapter file to get ID. Using stem. {e}")
    CHAPTER_ID = chapter_name
    
# Define Output Paths
SCENES_DIR = f"{OUTPUT_BASE}/scenes"
IMAGES_DIR = f"{OUTPUT_BASE}/images"
AUDIO_DIR = f"{OUTPUT_BASE}/audio"
VIDEO_DIR = f"{OUTPUT_BASE}/videos"

# Specific Artifact Paths (Expected location after running CLI)
SCENES_FILE = f"{SCENES_DIR}/ch{CHAPTER_ID:04d}_scenes.json"

print(f"Configuration Loaded:")
print(f"  Project Root: {PROJECT_ROOT_STR}")
print(f"  Input: {CHAPTER_FILE_PATH}")
print(f"  Chapter ID: {CHAPTER_ID}")
print(f"  Output Base: {OUTPUT_BASE}")
print(f"  Scenes File: {SCENES_FILE}")

## 1. Scene Extraction
Analyzes the novel text and splits it into visual scenes using Gemini.
**Input**: Chapter JSON
**Output**: Scenes JSON

In [ ]:
!python cli.py extract "{CHAPTER_FILE_PATH}" --output "{SCENES_FILE}"

## 2. Image Generation
Generates an image for each extracted scene using Pollinations.ai Flux.
**Input**: Scenes JSON
**Output**: PNG files in images directory

In [ ]:
!python cli.py images "{SCENES_FILE}" --output "{IMAGES_DIR}" --continue-on-error

## 3. Audio Generation (TTS)
Converts the text narration into speech audio files using Gemini TTS.
**Input**: Scenes JSON
**Output**: MP3 files in audio directory

In [ ]:
!python cli.py audio "{SCENES_FILE}" --output "{AUDIO_DIR}" --concurrent 3 --continue-on-error

## 4. Video Assembly
Combines images and audio into a final video file using FFmpeg with Ken Burns effect.
**Input**: Scenes, Images, Audio
**Output**: MP4 Video

In [ ]:
OUTPUT_VIDEO = f"{VIDEO_DIR}/chapter_{CHAPTER_ID:04d}.mp4"
!python cli.py video "{SCENES_FILE}" --images "{IMAGES_DIR}" --audio "{AUDIO_DIR}" --output "{OUTPUT_VIDEO}"
print(f"\n✅ Video created: {OUTPUT_VIDEO}")

---
## Alternative: Run Full Pipeline
Run all steps at once with a single command.

In [ ]:
PIPELINE_OUTPUT = f"{OUTPUT_BASE}/pipeline_ch{CHAPTER_ID:04d}"
!python cli.py pipeline "{CHAPTER_FILE_PATH}" --output "{PIPELINE_OUTPUT}"
print(f"\n✅ Pipeline complete! Check output in: {PIPELINE_OUTPUT}")